# 因子分析工作流：信号预测能力研究

这份 notebook 用于验证 `cta_lab` 中信号对未来收益的预测能力，形成第一版轻量因子研究工作流。

研究目标：
- 基于真实 `market_data` 构造连续价格矩阵
- 生成时序信号与截面 score
- 构造 future return labels
- 评估 IC / Rank IC / IR
- 比较不同信号、不同 horizon 的表现


## 1. 环境准备

这里统一导入数据层、signals operators、analysis.signal evaluator。后面如果要加新信号，可以直接沿用这个入口。


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
MARKET_DATA = ROOT.parent / "market_data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)


In [2]:
from data.loader import DataLoader, KlineSchema, ContractSchema
from data.sources.parquet_source import ParquetSource
from data.sources.column_keyed_source import ColumnKeyedSource

from signals.momentum import TSMOM, SharpeMomentum, AbsoluteMomentum
from signals.composite import RankCombiner
from signals.operators import rolling_zscore, clip, smooth, cross_sectional_rank

from analysis.signal import (
    build_forward_returns,
    evaluate_signal,
    information_coefficient,
)


## 2. 加载真实市场数据

这里使用国内期货真实 `market_data`，先把多个品种的连续价格拼成一个 `price_df`。

说明：
- `price_df` 是研究输入
- index 是交易日
- columns 是品种


In [3]:
loader = DataLoader(
    kline_source=ParquetSource(MARKET_DATA / "kline" / "china_daily_full"),
    kline_schema=KlineSchema.tushare(),
    contract_source=ColumnKeyedSource(
        MARKET_DATA / "contracts" / "china" / "contract_info.parquet",
        filter_col="fut_code",
    ),
    contract_schema=ContractSchema.tushare(),
)

symbols = ["RB", "HC", "CU", "AL", "AU", "M"]
price_df = pd.DataFrame({
    sym: loader.load_continuous(sym, adjust="nav", nav_output="price").prices
    for sym in symbols
}).dropna(how="all")

price_df.tail()


,RB,HC,CU,AL,AU,M
2026-03-23,3710.720014,8050.951375,99361.932356,18042.287334,763.126571,10867.895961
2026-03-24,3707.183750,8060.660090,101116.573995,18114.869054,753.234617,10752.318470
2026-03-25,3700.111221,8058.232911,101844.108334,18221.831587,768.234081,10633.129182
2026-03-26,3690.681183,8029.106768,102261.370675,18145.429777,775.833809,10658.411758
2026-03-27,3678.893636,8002.407804,102389.759087,18263.852582,763.234259,10593.399420


In [4]:
price_df.shape, price_df.index.min(), price_df.index.max()


((6980, 6), Timestamp('1997-07-02 00:00:00'), Timestamp('2026-03-27 00:00:00'))

## 3. 研究对象：构造几类 signal

这一步先做三类典型对象：
- `TSMOM(60)`：方向型时序信号
- `SharpeMomentum(60)`：连续强度型信号
- `RankCombiner([TSMOM(60)])`：截面排名 score

另外演示 `operators` 的使用，比如 `rolling_zscore`、`clip`、`smooth`。


In [5]:
tsmom_60 = price_df.apply(TSMOM(lookback=60).compute)
sharpe_60 = price_df.apply(SharpeMomentum(lookback=60).compute)
absmom_60 = price_df.apply(AbsoluteMomentum(lookback=60).compute)

sharpe_60_z = clip(rolling_zscore(sharpe_60, window=126), lower=-3.0, upper=3.0)
absmom_60_smooth = smooth(absmom_60, window=5, method="mean")
rank_score = RankCombiner([TSMOM(lookback=60)]).compute(price_df)
rank_score_cs = cross_sectional_rank(sharpe_60_z)

{
    "tsmom_60": tsmom_60.tail(2),
    "sharpe_60_z": sharpe_60_z.tail(2),
    "rank_score": rank_score.tail(2),
    "rank_score_cs": rank_score_cs.tail(2),
}


{'tsmom_60':              RB   HC   CU   AL   AU    M
 2026-03-26  1.0  1.0  1.0  1.0  1.0  1.0
 2026-03-27 -1.0  1.0  1.0  1.0 -1.0  1.0,
 'sharpe_60_z':                   RB        HC        CU        AL        AU         M
 2026-03-26  0.544314  0.634389 -2.134771 -1.287144 -2.187573  1.789332
 2026-03-27  0.309440  0.414484 -2.075004 -1.089934 -2.422460  1.602833,
 'rank_score':                   RB        HC        CU        AL        AU         M
 2026-03-26  0.583333  0.583333  0.583333  0.583333  0.583333  0.583333
 2026-03-27  0.250000  0.750000  0.750000  0.750000  0.250000  0.750000,
 'rank_score_cs':                   RB        HC        CU   AL        AU    M
 2026-03-26  0.666667  0.833333  0.333333  0.5  0.166667  1.0
 2026-03-27  0.666667  0.833333  0.333333  0.5  0.166667  1.0}

## 4. 构造 future return labels

这一步把未来收益标签统一下来。建议研究时始终明确 horizon，避免不同实验之间口径不一致。


In [6]:
forward_returns = build_forward_returns(price_df, horizons=[1, 5, 20, 60])
{k: v.tail(2) for k, v in forward_returns.items()}


{1:                   RB        HC        CU        AL       AU       M
 2026-03-26 -0.003194 -0.003325  0.001255  0.006526 -0.01624 -0.0061
 2026-03-27       NaN       NaN       NaN       NaN      NaN     NaN,
 5:             RB  HC  CU  AL  AU   M
 2026-03-26 NaN NaN NaN NaN NaN NaN
 2026-03-27 NaN NaN NaN NaN NaN NaN,
 20:             RB  HC  CU  AL  AU   M
 2026-03-26 NaN NaN NaN NaN NaN NaN
 2026-03-27 NaN NaN NaN NaN NaN NaN,
 60:             RB  HC  CU  AL  AU   M
 2026-03-26 NaN NaN NaN NaN NaN NaN
 2026-03-27 NaN NaN NaN NaN NaN NaN}

## 5. 评估截面预测能力：IC / Rank IC / IR

`evaluate_signal()` 目前按截面信号来评估：
- 输入：`signal_df(dates x symbols)`
- 对比：对应 horizon 的 future return 面板
- 输出：summary + 每个 horizon 的 IC / Rank IC 时间序列


In [7]:
report_rank = evaluate_signal(rank_score, price_df, horizons=[1, 5, 20, 60])
report_rank.summary


,ic_mean,ic_std,ic_ir,rank_ic_mean,rank_ic_std,rank_ic_ir,n_obs
horizon,,,,,,,
1,0.040214,0.606596,1.052381,0.034994,0.598377,0.928359,4732
5,0.053645,0.609165,1.397967,0.045525,0.599382,1.205708,4732
20,0.040665,0.616070,1.047823,0.036831,0.606074,0.964698,4720
60,0.045560,0.625461,1.156338,0.040034,0.623993,1.018481,4693


In [8]:
report_rank_cs = evaluate_signal(rank_score_cs, price_df, horizons=[1, 5, 20, 60])
report_rank_cs.summary


,ic_mean,ic_std,ic_ir,rank_ic_mean,rank_ic_std,rank_ic_ir,n_obs
horizon,,,,,,,
1,0.031378,0.651972,0.763998,0.028722,0.647752,0.703902,6788
5,0.037095,0.656807,0.896557,0.033693,0.648982,0.824143,6788
20,0.014325,0.653934,0.347734,0.008711,0.653088,0.211726,6773
60,-0.002526,0.656108,-0.061107,-0.008619,0.658041,-0.207932,6733


## 6. 把时序信号转成截面研究对象

时序信号如果要做截面预测测试，需要先在每个交易日横向比较不同品种。

一个常见做法是：
- 先算每个品种自己的强度信号
- 再做截面标准化，例如 zscore 或 rank
- 最后拿这个截面矩阵去评估对未来横截面收益的解释力


In [9]:
tsmom_strength = tsmom_60.replace(0.0, np.nan)
sharpe_rank = cross_sectional_rank(sharpe_60_z)
absmom_rank = cross_sectional_rank(absmom_60_smooth)

report_sharpe = evaluate_signal(sharpe_rank, price_df, horizons=[1, 5, 20, 60])
report_absmom = evaluate_signal(absmom_rank, price_df, horizons=[1, 5, 20, 60])

pd.concat({
    "rank_score": report_rank.summary["rank_ic_mean"],
    "sharpe_rank": report_sharpe.summary["rank_ic_mean"],
    "absmom_rank": report_absmom.summary["rank_ic_mean"],
}, axis=1)


,rank_score,sharpe_rank,absmom_rank
horizon,,,
1,0.034994,0.028722,0.022886
5,0.045525,0.033693,0.032499
20,0.036831,0.008711,0.019854
60,0.040034,-0.008619,0.022293


## 7. 看 IC 时间序列

summary 适合看整体均值，IC 时间序列适合观察稳定性。


In [10]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
report_rank.rank_ic_series[20].plot(ax=axes[0], title="Rank Score Rank IC (20d)")
report_sharpe.rank_ic_series[20].plot(ax=axes[1], title="Sharpe Rank Rank IC (20d)", color="tab:orange")
for ax in axes:
    ax.axhline(0.0, color="black", linewidth=1, linestyle="--")
    ax.grid(True, alpha=0.3)
plt.tight_layout()


## 8. 最小研究结论模板

每次做完一轮信号研究，建议至少记录：
- 用的 signal 定义
- 是否做了 operators 处理
- 测试的 horizons
- `IC mean / Rank IC mean / IR`
- 观察到的稳定性和失效区间

后面如果继续扩展，这份 notebook 可以继续补：
- quantile return
- factor return / long-short spread
- 多市场、多样本期对比
- 训练期 / 验证期拆分


In [11]:
research_summary = pd.concat(
    {
        "rank_score": report_rank.summary,
        "rank_score_cs": report_rank_cs.summary,
        "sharpe_rank": report_sharpe.summary,
        "absmom_rank": report_absmom.summary,
    },
    axis=1,
)
research_summary


rank_score                                                               rank_score_cs                      ... sharpe_rank  \
           ic_mean    ic_std     ic_ir rank_ic_mean rank_ic_std rank_ic_ir n_obs       ic_mean    ic_std     ic_ir  ... rank_ic_std   
horizon                                                                                                             ...               
1         0.040214  0.606596  1.052381     0.034994    0.598377   0.928359  4732      0.031378  0.651972  0.763998  ...    0.647752   
5         0.053645  0.609165  1.397967     0.045525    0.599382   1.205708  4732      0.037095  0.656807  0.896557  ...    0.648982   
20        0.040665  0.616070  1.047823     0.036831    0.606074   0.964698  4720      0.014325  0.653934  0.347734  ...    0.653088   
60        0.045560  0.625461  1.156338     0.040034    0.623993   1.018481  4693     -0.002526  0.656108 -0.061107  ...    0.658041   

                         absmom_rank                                                                
        rank_ic_ir n_obs     ic_mean    ic_std     ic_ir rank_ic_mean rank_ic_std rank_ic_ir n_obs  
horizon                                                                                             
1         0.703902  6788    0.025883  0.665500  0.617399     0.022886    0.659579   0.550813  6912  
5         0.824143  6788    0.037400  0.667636  0.889257     0.032499    0.657964   0.784098  6911  
20        0.211726  6773    0.024573  0.664756  0.586816     0.019854    0.656480   0.480090  6896  
60       -0.207932  6733    0.028967  0.662239  0.694373     0.022293    0.656581   0.538991  6856  

[4 rows x 28 columns]